| título | projeto | versão | data | autores | status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| CRISP-DM — Fase 4: Modeling | Projeção da Taxa de Congestionamento — Justiça Estadual (GO) | 1.0 | 14-12-2025 | Júlio César e Lays de Freitas | Rascunho |


Esse Notebook contém a *Modelagem Preditiva (Baseline)*.

### BIBLIOTECAS

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')


### IMPORTAÇÃO E CONFIGURAÇÃO

In [16]:
df_estatisticas_mes = pd.read_csv('datasets/df_estatisticas_mes.csv')

In [17]:

# Configurações visuais
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

df_full = df_estatisticas_mes.copy()

# Garantir que a coluna de data é datetime
df_full['mes_ref'] = pd.to_datetime(df_full['mes_ref'])
#df_full['Taxa de Congestionamento_mes'] = df_full['Taxa de Congestionamento_mes (%)'] * 100  # Converter para porcentagem

# Ordenar por unidade e data
df_full = df_full.sort_values(by=['comarca', 'serventia', 'mes_ref'])

print(f"Total de registros carregados: {len(df_full)}")
print(f"Período dos dados: de {df_full['mes_ref'].min().date()} até {df_full['mes_ref'].max().date()}")

Total de registros carregados: 571917
Período dos dados: de 2022-01-01 até 2025-10-01


In [18]:
df_full.sample(5)

,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%),mes_ref
218042,GOIÂNIA,4ª Vara da Fazenda Pública Estadual,226,1,218,99.54,2024-05-01
248527,VALPARAÍSO DE GOIÁS,"1ª Vara Cível (Cível, Infância e Juventude)",60,0,20,100.00,2024-07-01
366644,RIO VERDE,2ª Vara Cível,129,1,47,97.92,2025-01-01
527896,BOM JESUS DE GOIÁS,"1ª Vara Judicial (Família e Sucessões, Infânci...",98,9,69,88.46,2025-09-01
85033,SÃO MIGUEL DO ARAGUAIA,"2ª Vara Judicial (Fazendas Públicas, Criminal,...",77,13,9,40.91,2023-06-01


### REFINAMENTO DOS DADOS

In [19]:
# Vamos aplicar as regras de negócio para limpar a base antes do split temporal

# 1. Filtro de Tipo de Unidade: Remover CEJUSCs
# CEJUSCs costumam ter taxas de 0% ou 100% devido a mutirões
filtro_cejusc = ~df_full['serventia'].str.contains('CEJUSC|CENTRO JUDICIÁRIO', case=False, na=False)

# 2. Filtro de Volume (Tratamento de Zeros/Pequenas Amostras)
# Regra: Unidades com menos de 10 processos movimentados (Pendentes + Baixados) no mês são instáveis
# Se Pendentes=1 e Baixados=0 -> Taxa 100%. Se mês seguinte Pendentes=0 e Baixados=1 -> Taxa 0%.
df_full['volume_total'] = df_full['Pendentes_mes'] + df_full['Baixados_mes']
filtro_volume = df_full['volume_total'] >= 10

# Aplicação dos Filtros
df_refinado = df_full[filtro_cejusc & filtro_volume].copy()

# Agrupa por mês (usando o final do mês 'ME' ou início 'MS') e tira a média
df_refinado = df_refinado.groupby([
    'comarca', 
    'serventia', 
    pd.Grouper(key='mes_ref', freq='ME')  # Aqui ele entende que 'mes_ref' é o relógio
])['Taxa de Congestionamento_mes (%)'].mean().reset_index()

# Estatísticas do Refinamento
total_original = len(df_full)
total_refinado = len(df_refinado)
removidos = total_original - total_refinado

print(f"Registros Originais: {total_original}")
print(f"Registros Após Refinamento: {total_refinado}")
print(f"Registros Removidos (Ruído): {removidos} ({removidos/total_original:.1%} da base)")

Registros Originais: 571917
Registros Após Refinamento: 22135
Registros Removidos (Ruído): 549782 (96.1% da base)


In [20]:
df_refinado.sample(10)

,comarca,serventia,mes_ref,Taxa de Congestionamento_mes (%)
19083,TRIBUNAL DE JUSTIÇA,GABINETE DES. ITANEY FRANCISCO CAMPOS,2023-09-30,0.000000
2373,APARECIDA DE GOIÂNIA,"Vara da Fazenda Púb. Municipal, de Reg. Púb. e...",2023-04-30,99.659375
20306,TRIBUNAL DE JUSTIÇA,GABINETE DESA LÍLIA MÔNICA DE CASTRO BORGES ES...,2023-05-31,2.275000
21805,ÁGUAS LINDAS DE GOIÁS,"1ª Vara Cível, Família e Sucessões e da Infânc...",2024-06-30,93.345667
4914,FORMOSA,1ª Vara (Cível e da Inf. e da Juv.),2023-08-31,90.025625
19315,TRIBUNAL DE JUSTIÇA,GABINETE DES. JOSÉ RICARDO MARCOS MACHADO,2024-09-30,7.640000
15635,PONTALINA,Vara Judicial,2025-08-31,94.615227
20470,TRIBUNAL DE JUSTIÇA,GABINETE DESA. ALICE TELES DE OLIVEIRA,2025-10-31,100.000000
2620,ARUANÃ,Vara Judicial,2025-03-31,90.688919
14497,NIQUELÂNDIA,"Vara de Família, Sucessões, da Infância e da J...",2025-01-31,95.114595


### FEATURE ENGINEERING E DIVISÃO TEMPORAL

In [21]:
# Para Regressão Linear funcionar com datas, precisamos converter a data para um número (ordinal ou timestamp)
# 1. Criação da Feature Numérica de Tempo
df_refinado['mes_ordinal'] = df_refinado['mes_ref'].apply(lambda x: x.toordinal())

# 2. Definição do Ponto de Corte (Split Temporal)
# Vamos usar os últimos 3 meses disponíveis como validação (Teste) e o resto como Treino
data_maxima = df_refinado['mes_ref'].max()
data_corte = data_maxima - relativedelta(months=3)

print(f"Data de Corte para Validação: {data_corte.date()}")

# Separação
df_treino = df_refinado[df_refinado['mes_ref'] <= data_corte].copy()
df_validacao = df_refinado[df_refinado['mes_ref'] > data_corte].copy()

print(f"Registros de Treino (Histórico): {len(df_treino)}")
print(f"Registros de Validação (Recente): {len(df_validacao)}")

Data de Corte para Validação: 2025-07-31
Registros de Treino (Histórico): 20601
Registros de Validação (Recente): 1534


In [22]:
df_treino.to_csv('datasets/df_treino_congestionamento_regressãolinear.csv', index=False)

### TREINAMENTO DO MODELO (LOOP POR UNIDADE)

In [23]:
# Iremos iterar por cada serventia, treinar uma regressão linear individual e prever
resultados_validacao = []
modelos_dict = {} # Dicionário para guardar os modelos treinados para uso futuro

# Identificar todas as combinações únicas de Comarca e Serventia
unidades = df_refinado[['comarca', 'serventia']].drop_duplicates()

print(f"Iniciando treinamento para {len(unidades)} unidades jurisdicionais...")

for index, row in unidades.iterrows():
    comarca_atual = row['comarca']
    serventia_atual = row['serventia']
    
    # Filtrar dados da unidade específica
    mask_treino = (df_treino['comarca'] == comarca_atual) & (df_treino['serventia'] == serventia_atual)
    mask_valid = (df_validacao['comarca'] == comarca_atual) & (df_validacao['serventia'] == serventia_atual)
    
    dados_treino = df_treino[mask_treino]
    dados_valid = df_validacao[mask_valid]
    
    # Regra de negócio: Precisamos de pelo menos 2 pontos para traçar uma reta
    if len(dados_treino) >= 2:
        # Preparar X e y
        X_train = dados_treino[['mes_ordinal']]
        y_train = dados_treino['Taxa de Congestionamento_mes (%)']
        
        # Instanciar e Treinar
        modelo = LinearRegression()
        modelo.fit(X_train, y_train)
        
        # Guardar modelo
        modelos_dict[(comarca_atual, serventia_atual)] = modelo
        
        # Se houver dados de validação, fazer a previsão para avaliar performance
        if len(dados_valid) > 0:
            X_valid = dados_valid[['mes_ordinal']]
            y_real = dados_valid['Taxa de Congestionamento_mes (%)']
            
            # Previsão
            y_pred = modelo.predict(X_valid)
            
            # Armazenar resultados para avaliação
            temp_df = dados_valid.copy()
            temp_df['Taxa_Prevista'] = y_pred
            # Clipar previsões entre 0 e 100 (pois é taxa %)
            temp_df['Taxa_Prevista'] = temp_df['Taxa_Prevista'].clip(0, 100)
            
            resultados_validacao.append(temp_df)

# Consolidar resultados de validação
df_avaliado = pd.concat(resultados_validacao, ignore_index=True)
print("Treinamento e Validação concluídos.")

Iniciando treinamento para 513 unidades jurisdicionais...
Treinamento e Validação concluídos.


In [24]:
df_avaliado.to_csv('datasets/avaliacao_modelos_congestionamento_regressãolinear.csv', index=False)
#df_refinado.sample(10000).to_csv('datasets/full-refined-sample.csv', index=False)

### GERAÇÃO DE PREVISÕES FUTURAS

In [25]:
### GERAÇÃO DE PREVISÕES FUTURAS (3, 6 e 12 meses)

# Agora vamos projetar múltiplos horizontes temporais
horizontes = [3, 6, 12]  # meses
previsoes_futuras_por_horizonte = {horizon: [] for horizon in horizontes}

for (comarca, serventia), modelo in modelos_dict.items():
    for horizon in horizontes:
        # Gerar as datas futuras para cada horizonte
        datas_futuras = [data_maxima + relativedelta(months=i) for i in range(1, horizon + 1)]
        ordinais_futuros = np.array([d.toordinal() for d in datas_futuras]).reshape(-1, 1)
        
        # Prever para todas as datas do horizonte
        preds = modelo.predict(ordinais_futuros)
        
        for data, pred in zip(datas_futuras, preds):
            # Regra de negócio: Taxa não pode ser menor que 0 nem maior que 100
            pred_ajustado = max(0, min(100, pred))
            
            previsoes_futuras_por_horizonte[horizon].append({
                'comarca': comarca,
                'serventia': serventia,
                'data_futura': data,
                'horizonte_meses': horizon,
                'taxa_prevista': round(pred_ajustado, 2),
                'mes_no_horizonte': (data - data_maxima).days // 30  # Aproximação de meses
            })

In [26]:
# Consolidar resultados
dfs_futuros = {}
for horizon, previsoes in previsoes_futuras_por_horizonte.items():
    dfs_futuros[horizon] = pd.DataFrame(previsoes)

print("Amostra das previsões para diferentes horizontes:")
for horizon in [3, 6, 12]:
    print(f"\n=== Horizonte: {horizon} meses ===")
    display(dfs_futuros[horizon][dfs_futuros[horizon]['comarca'] == 'INHUMAS'].head())

Amostra das previsões para diferentes horizontes:

=== Horizonte: 3 meses ===


,comarca,serventia,data_futura,horizonte_meses,taxa_prevista,mes_no_horizonte
750,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2025-11-30,3,93.68,1
751,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2025-12-31,3,93.96,2
752,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2026-01-31,3,94.24,3
753,INHUMAS,"Vara Cível, Infância e Juventude e Juizado Esp...",2025-11-30,3,92.82,1
754,INHUMAS,"Vara Cível, Infância e Juventude e Juizado Esp...",2025-12-31,3,93.05,2



=== Horizonte: 6 meses ===


,comarca,serventia,data_futura,horizonte_meses,taxa_prevista,mes_no_horizonte
1500,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2025-11-30,6,93.68,1
1501,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2025-12-31,6,93.96,2
1502,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2026-01-31,6,94.24,3
1503,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2026-02-28,6,94.50,4
1504,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2026-03-31,6,94.78,5



=== Horizonte: 12 meses ===


,comarca,serventia,data_futura,horizonte_meses,taxa_prevista,mes_no_horizonte
3000,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2025-11-30,12,93.68,1
3001,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2025-12-31,12,93.96,2
3002,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2026-01-31,12,94.24,3
3003,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2026-02-28,12,94.50,4
3004,INHUMAS,"Vara Criminal (crime em geral, crimes dolosos ...",2026-03-31,12,94.78,5


In [27]:
display(dfs_futuros[3][dfs_futuros[3]['comarca'] == 'GOIÂNIA'])

,comarca,serventia,data_futura,horizonte_meses,taxa_prevista,mes_no_horizonte
411,GOIÂNIA,10ª Vara Cível,2025-11-30,3,93.11,1
412,GOIÂNIA,10ª Vara Cível,2025-12-31,3,93.22,2
413,GOIÂNIA,10ª Vara Cível,2026-01-31,3,93.34,3
414,GOIÂNIA,10º Juizado Especial Cível,2025-11-30,3,87.61,1
415,GOIÂNIA,10º Juizado Especial Cível,2025-12-31,3,89.45,2
...,...,...,...,...,...,...
730,GOIÂNIA,Gabinete da Central de Cumprimento de Sentença...,2025-12-31,3,100.00,2
731,GOIÂNIA,Gabinete da Central de Cumprimento de Sentença...,2026-01-31,3,100.00,3
732,GOIÂNIA,Gabinete da Central de Cumprimento de Sentença...,2025-11-30,3,100.00,1
733,GOIÂNIA,Gabinete da Central de Cumprimento de Sentença...,2025-12-31,3,100.00,2


In [28]:
dfs_futuros[12].to_csv('results/previsoes_futuras_congestionamento_12_meses.csv', index=False)
dfs_futuros[3].to_csv('results/previsoes_futuras_congestionamento_3_meses.csv', index=False)
dfs_futuros[6].to_csv('results/previsoes_futuras_congestionamento_6_meses.csv', index=False)
#!/usr/bin/env python3